In [42]:
# 01 패키지
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from pathlib import Path

In [43]:
# 02 랜덤 고정값
np.random.seed(42)
random.seed(42)

In [44]:
# 03 경로 설정
DATA_DIR = Path("../data")
OUTPUT_PATH = DATA_DIR / "18_campaign_influencers.csv"

In [45]:
# 04 campaigns 데이터 불러오기
campaigns = pd.read_csv(DATA_DIR / "02_campaigns.csv", encoding="utf-8-sig")

# 컬럼명 앞뒤 공백 제거
campaigns.columns = campaigns.columns.str.strip()

# 빈 문자열을 NaN으로 변환
campaigns = campaigns.replace(r"^\s*$", np.nan, regex=True)

# 전체가 빈 행인 경우 제거
campaigns = campaigns.dropna(how="all")

# campaign_id가 없는 행 제거
campaigns = campaigns.dropna(subset=["campaign_id"])

# 인덱스 재정렬
campaigns = campaigns.reset_index(drop=True)

# 날짜 컬럼 변환
campaigns["campaign_start_at"] = pd.to_datetime(campaigns["campaign_start_at"], errors="coerce")
campaigns["campaign_end_at"] = pd.to_datetime(campaigns["campaign_end_at"], errors="coerce")

# 데이터 확인
print(campaigns.shape)
campaigns.head()

(3, 6)


,campaign_id,client_id,campaign_name,campaign_start_at,campaign_end_at,campaign_status
0,cam-0001,cli-0001,[라이언스윔] ROAR 남성 수영복 세트 (수영모+수영복+수경) 체험단 모집,2026-05-16 10:00:00,2026-05-30 23:59:00,종료
1,cam-0002,cli-0001,[라이언스윔] ROAR 여성 수영복 세트 (수영모+수영복+수경) 체험단 모집,2026-05-16 10:00:00,2026-05-30 23:59:00,종료
2,cam-0003,cli-0001,[라이언스윔] ROAR 수영 샴푸,2026-05-29 10:00:00,2026-06-12 23:59:00,종료


In [46]:
# 05 influencers 데이터 불러오기
influencers = pd.read_csv(DATA_DIR / "04_influencers.csv", encoding="utf-8-sig")

# 컬럼명 앞뒤 공백 제거
influencers.columns = influencers.columns.str.strip()

# 빈 문자열을 NaN으로 변환
influencers = influencers.replace(r"^\s*$", np.nan, regex=True)

# 전체가 빈 행인 경우 제거
influencers = influencers.dropna(how="all")

# influencer_id가 없는 행 제거
influencers = influencers.dropna(subset=["influencer_id"])

# 인덱스 재정렬
influencers = influencers.reset_index(drop=True)

# 데이터 확인
print(influencers.shape)
influencers.head()

(200, 12)


,influencer_id,influencer_name,influencer_created_at,influencer_registration_type,influencer_platform,influencer_category,influencer_followers_count,influencer_posts,influencer_avg_likes,influencer_avg_comments,influencer_engagement_rate,influencer_recent_post_at
0,Inf-0001,zrathbourne0,2025-12-02 09:57:47,스포츠크루 및 커뮤니티,Instagram,바레,65010,71,302.49,241.75,0.84,2026-02-17 11:09:28
1,Inf-0002,kalred1,2025-10-04 15:03:03,개인 인플루언서,Instagram,크로스핏,90294,159,805.09,331.81,1.26,2026-04-12 02:27:17
2,Inf-0003,lelgey2,2025-07-03 21:59:09,웰니스 피트니스 센터,YouTube,요가,75232,250,239.19,228.80,0.62,2026-02-17 09:38:13
3,Inf-0004,hcasson3,2025-10-25 05:41:08,웰니스 피트니스 센터,YouTube,요가,53117,270,405.20,178.78,1.10,2026-02-26 01:28:20
4,Inf-0005,lmangam4,2025-11-11 03:06:02,스포츠크루 및 커뮤니티,YouTube,러닝,55393,221,619.50,528.73,2.07,2026-04-10 05:26:51


In [47]:
# 06 campaign_influencers 데이터 생성

campaign_influencer_rows = []
campaign_influencer_id = 1

for _, campaign in campaigns.iterrows():
    campaign_id = campaign["campaign_id"]
    campaign_start_at = campaign["campaign_start_at"]

    # 캠페인 1개당 지원 인플루언서 20~30명
    applicant_count = np.random.randint(20, 30)
    applicant_count = min(applicant_count, len(influencers))

    selected_influencers = influencers.sample(
        n=applicant_count,
        replace=False,
        random_state=campaign_influencer_id
    ).reset_index(drop=True)

    # 지원자 중 선정 인원
    # 최소 10명, 최대 지원자의 60% 정도 선정
    min_selected = min(10, applicant_count)
    max_selected = max(min_selected, int(applicant_count * 0.6))

    selected_count = np.random.randint(min_selected, max_selected + 1)

    selected_index_list = random.sample(
        list(selected_influencers.index),
        selected_count
    )

    status_list = []

    for idx in selected_influencers.index:
        if idx in selected_index_list:
            # 선정된 사람 중 일부는 콘텐츠 등록 완료까지 감
            status = np.random.choice(
                ["선정", "콘텐츠 등록 완료"],
                p=[0.20, 0.80]
            )
        else:
            # 선정되지 않은 지원자
            status = "지원"

        status_list.append(status)

    # 캠페인마다 최소 1명은 완료 상태가 나오도록 보정
    if "콘텐츠 등록 완료" not in status_list:
        first_selected_idx = selected_index_list[0]
        status_list[first_selected_idx] = "콘텐츠 등록 완료"

    for idx, influencer in selected_influencers.iterrows():
        influencer_id = influencer["influencer_id"]
        participation_status = status_list[idx]

        # 지원일: 캠페인 시작 시각 이후 ~ 캠페인 시작 6일 후까지
        # 예: campaign_start_at이 2026-05-16 10:00이면
        # 지원일은 2026-05-16 10:00 이후부터 생성됨
        applied_start = campaign_start_at
        applied_end = campaign_start_at + timedelta(days=6)

        applied_seconds = np.random.randint(
            0,
            int((applied_end - applied_start).total_seconds()) + 1
        )

        applied_at = applied_start + timedelta(seconds=int(applied_seconds))

        # 선정일: 선정/콘텐츠 등록 완료인 경우에만 생성
        if participation_status in ["선정", "콘텐츠 등록 완료"]:
            # selected_at = applied_at + timedelta(days=np.random.randint(1, 6))
            selected_at = campaign_start_at + timedelta(days=7)
        else:
            selected_at = pd.NaT

        # 지급금액
        # 실제 리워드 기준이 없으므로 랜덤 금액을 사용하지 않고,
        # 콘텐츠 등록 완료자에게만 기본 리워드 20,000원을 지급하는 것으로 설정

        if participation_status == "콘텐츠 등록 완료":
            reward_amount = 20000
        else:
            reward_amount = 0

        campaign_influencer_rows.append({
            "campaign_influencer_id": f"caminf-{campaign_influencer_id:04d}",
            "influencer_id": influencer_id,
            "campaign_id": campaign_id,
            "applied_at": applied_at.strftime("%Y-%m-%d %H:%M:%S"),
            "selected_at": selected_at.strftime("%Y-%m-%d %H:%M:%S") if pd.notna(selected_at) else "",
            "participation_status": participation_status,
            "reward_amount": reward_amount
        })

        campaign_influencer_id += 1

campaign_influencers = pd.DataFrame(campaign_influencer_rows)

print(campaign_influencers.shape)
campaign_influencers.head()

(79, 7)


,campaign_influencer_id,influencer_id,campaign_id,applied_at,selected_at,participation_status,reward_amount
0,caminf-0001,Inf-0059,cam-0001,2026-05-19 12:17:35,2026-05-23 10:00:00,콘텐츠 등록 완료,20000
1,caminf-0002,Inf-0041,cam-0001,2026-05-17 14:42:35,,지원,0
2,caminf-0003,Inf-0035,cam-0001,2026-05-19 03:29:56,2026-05-23 10:00:00,콘텐츠 등록 완료,20000
3,caminf-0004,Inf-0103,cam-0001,2026-05-18 21:29:36,2026-05-23 10:00:00,콘텐츠 등록 완료,20000
4,caminf-0005,Inf-0185,cam-0001,2026-05-18 13:19:39,2026-05-23 10:00:00,선정,0


In [48]:
# 08 CSV 저장

campaign_influencers.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"저장 완료: {OUTPUT_PATH}")

저장 완료: ..\data\18_campaign_influencers.csv
